# 07 · Inverses and the pseudoinverse

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/07-inverses-and-pseudoinverse.ipynb)

*Part IV · exercise · 15 min*

> 🇪🇸 **Inversas y la pseudoinversa** — Resolver un sistema de 20.433 ecuaciones que no tiene solución exacta.

Solve a 20,433-equation system that has no exact solution.

## What you will be able to do

- Say when a square matrix has no inverse, and predict the error before you see it.
- Compute the Moore-Penrose pseudoinverse and verify its four defining conditions.
- Say what `x = A⁺b` gives you for a tall matrix and for a wide one.
- Solve a real 20,433-equation system that has no exact solution.
- Apply the pseudoinverse to a tensor by unfolding, solving, and folding back.

## Setup

Run this first. It installs and imports everything this notebook needs, and nothing else.

> 🇪🇸 Ejecuta esto primero: instala e importa todo lo que este cuaderno necesita.

In [ ]:
import numpy as np
import pandas as pd

HOUSING = "https://raw.githubusercontent.com/ageron/handson-ml2/master/datasets/housing/housing.csv"
housing = pd.read_csv(HOUSING)

def unfold(T, axis):
    return np.moveaxis(T, axis, 0).reshape(T.shape[axis], -1)

rng = np.random.default_rng(0)
print(housing.shape)                                   # (20640, 10)
print(housing['total_bedrooms'].isnull().sum())        # 207 missing values!

## Step 1 — square matrices

> 🇪🇸 Paso 1: matrices cuadradas. La inversa solo existe si las columnas son
> linealmente independientes.

Chapter 2 §2.3 defines `A⁻¹` for a square matrix, with `A⁻¹A = I`. But this only
exists when the columns are linearly independent. A matrix with dependent
columns is **singular** and has no inverse.

In [ ]:
S = np.array([[2., 1.], [1., 3.]])
print(np.round(np.linalg.inv(S) @ S, 12))    # the identity, fine

Singular = np.array([[1., 2.], [2., 4.]])    # column 2 = 2 x column 1
try:
    np.linalg.inv(Singular)
except np.linalg.LinAlgError as e:
    print("LinAlgError:", e)                 # this error is the expected result

## Step 2 — non-square matrices

> 🇪🇸 Paso 2: matrices no cuadradas. `A⁻¹` ni siquiera está definida, pero la
> pseudoinversa sí.

`A⁻¹` is not even defined. But we still need to solve `Ax = b`, and in machine
learning `A` is almost never square: it has one row per example and one column
per feature, and there are always far more examples than features.

The **Moore-Penrose pseudoinverse** `A⁺` (Chapter 2 §2.9) is the answer. It is
defined for *every* matrix — square or not, singular or not — and it is computed
from the SVD (eq. 2.47):

$$A^{+} = V D^{+} U^{\top}$$

In [ ]:
A = rng.standard_normal((5, 3))
A_plus = np.linalg.pinv(A)
print(A.shape, A_plus.shape)               # (5, 3) (3, 5) — note the shape flips

U, S_, Vt = np.linalg.svd(A, full_matrices=False)
print(np.allclose(A_plus, Vt.T @ np.diag(1 / S_) @ U.T))   # True — this is eq 2.47

It satisfies four conditions that define it uniquely.

In [ ]:
print(np.allclose(A @ A_plus @ A, A))            # 1
print(np.allclose(A_plus @ A @ A_plus, A_plus))  # 2
print(np.allclose((A @ A_plus).T, A @ A_plus))   # 3
print(np.allclose((A_plus @ A).T, A_plus @ A))   # 4

What `A⁺` gives you depends on the shape, exactly as Chapter 2 §2.9 says:

- **More rows than columns** (too many equations, usually no exact solution) →
  `x = A⁺b` gives the `x` that makes `Ax` as **close as possible** to `b`.
  This is least squares.
- **More columns than rows** (too few equations, infinitely many solutions) →
  `x = A⁺b` gives the valid solution with the **smallest norm**.

## Step 3 — what about tensors?

> 🇪🇸 Paso 3: ¿y los tensores? No hay una única inversa tensorial aceptada por
> todos. En la práctica se despliega, se resuelve como matriz y se vuelve a
> plegar.

This is a fair question with an honest answer. There is no single tensor inverse
that everyone uses. Several definitions exist (based on the Einstein product, or
the t-product for order-3 tensors), and they are active research.

**In practice, in machine learning, you unfold the tensor into a matrix, use the
matrix pseudoinverse, and fold the result back.** That works because unfolding
loses nothing — which you proved for yourself in section 01.

In [ ]:
T = rng.standard_normal((4, 3, 5))
M = unfold(T, 0)                        # (4, 15)
M_plus = np.linalg.pinv(M)              # (15, 4)
print(M.shape, M_plus.shape)
print(np.allclose(M @ M_plus @ M, M))   # True

**When a tensor problem is hard, unfold it to a matrix, solve it there, and
fold back.** That is a general lesson, and section 10 is built entirely on it.

## Exercise 1 — real California housing data

> 🇪🇸 Datos reales de vivienda en California: 20.640 distritos censales.

Predict house value from district features. 20,640 real districts, 207 of them
with a missing value.

::: {.callout-note}
TODO 3 asks you to trigger an error on purpose. If it raises, you did it right.
:::

In [ ]:
# TODO 1: Drop rows with missing values. How many rows remain?

# TODO 2: Build X from these columns, and add a column of ones for the bias:
#         ['housing_median_age','total_rooms','total_bedrooms',
#          'population','households','median_income']
#         Target y = 'median_house_value'. Print X.shape. Is X square?

# TODO 3: Try np.linalg.inv(X). What happens, and why?
#         THE ERROR IS THE EXPECTED RESULT — you have not done anything wrong.

In [ ]:
#@title Solution — try it yourself first { display-mode: 'form' }
d = housing.dropna()
print(len(d))                                            # 20433 rows remain

feats = ['housing_median_age','total_rooms','total_bedrooms',
         'population','households','median_income']
X = np.column_stack([np.ones(len(d)), d[feats].to_numpy(float)])   # (20433, 7)
y = d['median_house_value'].to_numpy(float)
print(X.shape)                                           # (20433, 7) — very tall

try:
    np.linalg.inv(X)
except np.linalg.LinAlgError as e:
    print("LinAlgError:", e)   # inv() requires a SQUARE matrix. X has 20,433
                               # rows and 7 columns, so it cannot even be called.

## Exercise 2 — solve it anyway

> 🇪🇸 Resuélvelo de todas formas, con la pseudoinversa.

In [ ]:
# TODO 4: Solve for the weights with the pseudoinverse: w = pinv(X) @ y.

# TODO 5: Check your answer against np.linalg.lstsq. Do they agree?

# TODO 6: Compute the RMSE of the predictions. Which feature has the largest
#         coefficient, and does that make sense for house prices?

In [ ]:
#@title Solution — try it yourself first { display-mode: 'form' }
w = np.linalg.pinv(X) @ y
w_lstsq, *_ = np.linalg.lstsq(X, y, rcond=None)
print(np.allclose(w, w_lstsq))                            # True

rmse = np.sqrt(((X @ w - y) ** 2).mean())
print(round(rmse))                                        # ~75980

coef, name = max(zip(w[1:], feats))
print(name, round(coef))                                  # median_income 47748

# X is 20433 x 7 — very tall, so np.linalg.inv cannot even be called. There is
# NO EXACT SOLUTION: no straight line passes through 20,433 points. The
# pseudoinverse gives the best possible answer instead, and lstsq agrees exactly
# because it solves the same problem.
#
# The largest coefficient belongs to median_income, which is the sensible
# result — income predicts house prices.

import matplotlib.pyplot as plt
pred = X @ w
residuals = pred - y
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
axes[0].scatter(y, pred, s=3, alpha=0.2, color="#4C72B0")
lims = [min(y.min(), pred.min()), max(y.max(), pred.max())]
axes[0].plot(lims, lims, color="#C44E52", linewidth=1)
axes[0].set_xlabel("actual"); axes[0].set_ylabel("predicted")
axes[0].set_title("predicted vs actual")
axes[1].hist(residuals, bins=60, color="#55A868")
axes[1].set_xlabel("prediction - actual"); axes[1].set_title("residuals")
plt.tight_layout()
plt.show()

# No straight line fits 20,433 points exactly, and the residuals show it: they
# are not tightly clustered at zero, and the predicted-vs-actual scatter fans
# out badly at the high end. LEAST SQUARES MINIMIZES THE AVERAGE SQUARED ERROR
# ACROSS ALL POINTS — it says nothing about any one prediction being close.

## Exercise 3 — the tensor version

> 🇪🇸 La versión tensorial: despliega, resuelve, vuelve a plegar.

In [ ]:
# TODO 7: Take an order-3 tensor T of shape (4, 3, 5) and a vector b of
#         length 4. Solve the unfolded least-squares problem for x, then fold
#         x back to the shape of a mode-0 slice. What shape must x have?

In [ ]:
#@title Solution — try it yourself first { display-mode: 'form' }
T = rng.standard_normal((4, 3, 5))
b = rng.standard_normal(4)

M = unfold(T, 0)                       # (4, 15) — one row per index along axis 0
x_flat = np.linalg.pinv(M) @ b         # (15,)   — min-norm solution, wide matrix
x = x_flat.reshape(T.shape[1], T.shape[2])       # fold back to (3, 5)
print(M.shape, x_flat.shape, x.shape)

print(np.allclose(M @ x_flat, b))      # True — 4 equations, 15 unknowns

# M is WIDE (4 x 15): infinitely many solutions, and pinv picks the one with the
# smallest norm. Folding back to (3, 5) is only meaningful because unfolding
# lost nothing in the first place.

---

## Time for Kahoot 🎯

**Kahoot 2 — Einsum, Distance & the Pseudoinverse** · 6 questions, about 5 minutes.

> 🇪🇸 **Einsum, distancia y la pseudoinversa** — 6 preguntas, unos 5 minutos.

Join at **kahoot.it** with the PIN on the facilitator's screen.

- [Quiz details and facilitator notes](https://project-delphi.github.io/tensors-workshop/kahoot.html#quiz-2)
- [Import file (`.xlsx`)](https://github.com/project-delphi/tensors-workshop/blob/main/kahoot/kahoot_quiz_2_distance_pseudoinverse.xlsx)

Next up: **08 · Recursion with matrices and vectors** — [open in Colab](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/08-recursion-with-matrices.ipynb).

[← Back to the workshop site](https://project-delphi.github.io/tensors-workshop/) · [All notebooks](https://project-delphi.github.io/tensors-workshop/notebooks.html) · [Handbook](https://project-delphi.github.io/tensors-workshop/tensors_workshop_plan_with_quizzes.html)